# DeiT-Tiny PYNQ Inference v1 (No Torch/Timm)

PL accelerates GEMM; PS runs LN/Softmax/GELU/Residual.
All prints are ASCII.


In [35]:
print('DeiT PYNQ Inference v2 (double-buffer + async pack)')
import os
import time
import numpy as np
import pynq


DeiT PYNQ Inference v2 (double-buffer + async pack)


In [36]:
# Paths (relative to this notebook)
NB_DIR = os.getcwd()
BASE = os.path.abspath(os.path.join(NB_DIR, '..'))
BITSTREAM = os.path.join(BASE, 'deit', 'deit_accel.bit')
WEIGHT_DIR = os.path.join(BASE, 'deit', 'weights_npy')
IMAGE_PATH = os.path.join(BASE, 'deit', 'Puppy-Cover.jpg')
IMAGE_NPY = os.path.join(BASE, 'deit', 'image_fp32.npy')

USE_IMAGE_NPY = False  # True if you preprocessed image with preprocess_image_npy.py

ARRAY_ROW = 12
ARRAY_COL = 16
ACT_RANGE = 3.0
ACT_TARGET = 64
MAX_BLOCKS = 12
TIMEOUT = 3.0

print('[INFO] BITSTREAM =', BITSTREAM)
print('[INFO] WEIGHT_DIR =', WEIGHT_DIR)
print('[INFO] IMAGE_PATH =', IMAGE_PATH)
print('[INFO] IMAGE_NPY =', IMAGE_NPY)


[INFO] BITSTREAM = /home/xilinx/jupyter_notebooks/ps/deit/deit_accel.bit
[INFO] WEIGHT_DIR = /home/xilinx/jupyter_notebooks/ps/deit/weights_npy
[INFO] IMAGE_PATH = /home/xilinx/jupyter_notebooks/ps/deit/Puppy-Cover.jpg
[INFO] IMAGE_NPY = /home/xilinx/jupyter_notebooks/ps/deit/image_fp32.npy


In [37]:
def ceil_to(x, base):
    return ((x + base - 1) // base) * base

def calc_pad(m, k, n):
    m_pad = m if (m % 2 == 0) else (m + 1)
    k_pad = ceil_to(k, ARRAY_ROW)
    n_pad = ceil_to(n, ARRAY_COL)
    return m_pad, k_pad, n_pad

def to_u8(x):
    return int(x) & 0xFF

def u8_to_i8(arr_u8):
    arr_u8 = np.array(arr_u8, dtype=np.uint8)
    return np.where(arr_u8 < 128, arr_u8, arr_u8 - 256).astype(np.int8)

def pack_input_tile(a_tile):
    bytes_list = []
    for r in range(a_tile.shape[0]):
        for c in range(ARRAY_ROW):
            bytes_list.append(to_u8(a_tile[r, c]))
    words = []
    for i in range(0, len(bytes_list), 8):
        w = 0
        for b_i in range(8):
            w |= (bytes_list[i + b_i] << (8 * b_i))
        words.append(np.uint64(w))
    return words

def pack_weight_tile(b_tile):
    words = []
    for r in range(ARRAY_ROW):
        val128 = 0
        for c in range(ARRAY_COL):
            val = to_u8(b_tile[r, c])
            val128 |= (val << (8 * c))
        low = val128 & 0xFFFFFFFFFFFFFFFF
        high = (val128 >> 64) & 0xFFFFFFFFFFFFFFFF
        words.append(np.uint64(low))
        words.append(np.uint64(high))
    return words

def unpack_output_tile(words, m_pad):
    out = np.zeros((m_pad, ARRAY_COL), dtype=np.int8)
    for r in range(m_pad):
        low = int(words[2*r])
        high = int(words[2*r + 1])
        val128 = low | (high << 64)
        bytes_row = [(val128 >> (8*c)) & 0xFF for c in range(ARRAY_COL)]
        out[r, :] = u8_to_i8(bytes_row)
    return out

def layernorm_ps(x, w, b, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    y = (x - mean) / np.sqrt(var + eps)
    return y * w + b

def gelu_ps(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * (x ** 3))))

def softmax_ps(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)

def quantize_act(x_fp32):
    scale = ACT_RANGE / float(ACT_TARGET)
    q = np.rint(x_fp32 / scale)
    q = np.clip(q, -127, 127).astype(np.int8)
    return q, scale

def dequantize_int8(x_int8, scale):
    return x_int8.astype(np.float32) * scale

def compute_ppu_params(ratio):
    if ratio <= 0:
        return 0, 0
    best = (1, 0, 1.0)
    for shift in range(0, 32):
        mult = int(round(ratio * (1 << shift)))
        if mult <= 0 or mult > 32767:
            continue
        err = abs(ratio - (mult / float(1 << shift)))
        if err < best[2]:
            best = (mult, shift, err)
    return best[0], best[1]


In [38]:
# AXI-Lite regs
REG_CTRL      = 0x00
REG_STATUS    = 0x04
REG_CFG_SEQ   = 0x08
REG_CFG_ACC   = 0x0C
REG_VERSION   = 0x10
REG_PPU_MULT  = 0x14
REG_PPU_SHIFT = 0x18
REG_PPU_ZP    = 0x1C
REG_PPU_BIAS  = 0x20
REG_OUT_EN    = 0x24

REG_DBG_SNAP  = 0x28
REG_DBG0      = 0x30

overlay = pynq.Overlay(BITSTREAM)
axi_ctrl = overlay.deit_accelerator_top_0
dma = overlay.axi_dma_0
print('[INFO] Overlay loaded')
print('VERSION = 0x%08x' % axi_ctrl.read(REG_VERSION))

def axi_write(addr, val):
    axi_ctrl.write(addr, int(val))

def axi_read(addr):
    return axi_ctrl.read(addr)

def soft_reset():
    axi_write(REG_CTRL, 0x00)
    time.sleep(0.01)
    axi_write(REG_CTRL, 0x02)
    time.sleep(0.01)

def start_pulse():
    axi_write(REG_CTRL, 0x03)
    axi_write(REG_CTRL, 0x02)

def dbg_snap():
    axi_write(REG_DBG_SNAP, 1)

def dbg_read_dma_req():
    dbg_snap()
    d0 = axi_read(REG_DBG0)
    return (d0 >> 3) & 0x1

def dma_status(ch):
    return ch._mmio.read(0x04)

def wait_dma_idle(ch, timeout=2.0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        sr = dma_status(ch)
        if sr & 0x2:
            return True, sr
        time.sleep(0.001)
    return False, dma_status(ch)


[INFO] Overlay loaded
VERSION = 0x20260117


In [39]:
def run_gemm_pl_v2(a_int8, b_int8, scale_a, scale_b, scale_out, timeout=3.0):
    """
    PS ???? + ?????? PL ???????? tile?
    ???? axi_ctrl / dma?
    """
    m, k = a_int8.shape
    k2, n = b_int8.shape
    assert k2 == k
    m_pad, k_pad, n_pad = calc_pad(m, k, n)
    k_tiles = k_pad // ARRAY_ROW
    n_tiles = n_pad // ARRAY_COL

    a_pad = np.zeros((m_pad, k_pad), dtype=np.int8)
    b_pad = np.zeros((k_pad, n_pad), dtype=np.int8)
    a_pad[:m, :k] = a_int8
    b_pad[:k, :n] = b_int8

    ratio = (scale_a * scale_b) / scale_out
    mult, shift = compute_ppu_params(ratio)

    soft_reset()
    axi_write(REG_CFG_SEQ, m_pad)
    axi_write(REG_CFG_ACC, 0)
    axi_write(REG_PPU_MULT, mult)
    axi_write(REG_PPU_SHIFT, shift)
    axi_write(REG_PPU_ZP, 0)
    axi_write(REG_PPU_BIAS, 0)
    axi_write(REG_OUT_EN, 0)

    c_hw = np.zeros((m_pad, n_pad), dtype=np.int8)

    in_words_len = (m_pad * ARRAY_ROW) // 8
    w_words_len = ARRAY_ROW * 2
    out_words_len = m_pad * 2

    buf_A = [pynq.allocate(shape=(in_words_len,), dtype=np.uint64),
             pynq.allocate(shape=(in_words_len,), dtype=np.uint64)]
    buf_B = [pynq.allocate(shape=(w_words_len,), dtype=np.uint64),
             pynq.allocate(shape=(w_words_len,), dtype=np.uint64)]
    buf_C = [pynq.allocate(shape=(out_words_len,), dtype=np.uint64),
             pynq.allocate(shape=(out_words_len,), dtype=np.uint64)]

    def prep_tile_to_buf(n_idx, k_idx, buf_idx):
        a_tile = a_pad[:, k_idx*ARRAY_ROW:(k_idx+1)*ARRAY_ROW]
        b_tile = b_pad[k_idx*ARRAY_ROW:(k_idx+1)*ARRAY_ROW, n_idx*ARRAY_COL:(n_idx+1)*ARRAY_COL]
        in_words = pack_input_tile(a_tile)
        w_words = pack_weight_tile(b_tile)
        buf_A[buf_idx][:] = in_words
        buf_B[buf_idx][:] = w_words
        buf_A[buf_idx].flush()
        buf_B[buf_idx].flush()

    try:
        tiles = [(n_idx, k_idx) for n_idx in range(n_tiles) for k_idx in range(k_tiles)]
        if len(tiles) == 0:
            return c_hw[:m, :n]

        # ?????? tile
        prep_tile_to_buf(tiles[0][0], tiles[0][1], buf_idx=0)

        for t_idx, (n_idx, k_idx) in enumerate(tiles):
            buf_idx = t_idx % 2
            next_idx = (t_idx + 1) % 2

            acc_mode = 0 if (k_idx == 0) else 1
            out_en = 1 if (k_idx == k_tiles - 1) else 0
            axi_write(REG_CFG_ACC, acc_mode)
            axi_write(REG_OUT_EN, out_en)

            # 1) ???? A
            dma.sendchannel.transfer(buf_A[buf_idx])
            ok_mm2s, _ = wait_dma_idle(dma.sendchannel, timeout=timeout)
            if not ok_mm2s:
                raise RuntimeError('MM2S not idle after A preload')

            # 2) ????????? S2MM
            if out_en:
                dma.recvchannel.transfer(buf_C[buf_idx])

            # 3) ?? PL
            start_pulse()

            # 4) ?? DMA_REQ ????? B
            t0 = time.time()
            while time.time() - t0 < timeout:
                if dbg_read_dma_req() == 1:
                    break
                time.sleep(0.0005)
            else:
                raise RuntimeError('DMA_REQ timeout')

            dma.sendchannel.transfer(buf_B[buf_idx])
            ok_mm2s2, _ = wait_dma_idle(dma.sendchannel, timeout=timeout)
            if not ok_mm2s2:
                raise RuntimeError('MM2S not idle after weight')

            # 5) ? PL ???????? tile
            if t_idx + 1 < len(tiles):
                n2, k2 = tiles[t_idx + 1]
                prep_tile_to_buf(n2, k2, buf_idx=next_idx)

            # 6) ?? AP_DONE
            t0 = time.time()
            done = False
            while time.time() - t0 < timeout:
                if (axi_read(REG_STATUS) & 0x1) != 0:
                    done = True
                    axi_write(REG_STATUS, 0x1)
                    break
                time.sleep(0.0005)
            if not done:
                raise RuntimeError('AP_DONE timeout')

            # 7) ????
            if out_en:
                ok_s2mm, _ = wait_dma_idle(dma.recvchannel, timeout=timeout)
                if not ok_s2mm:
                    raise RuntimeError('S2MM timeout')
                buf_C[buf_idx].invalidate()
                out_tile = unpack_output_tile(np.array(buf_C[buf_idx]), m_pad)
                c_hw[:, n_idx*ARRAY_COL:(n_idx+1)*ARRAY_COL] = out_tile
    finally:
        for b in buf_A + buf_B + buf_C:
            b.close()

    return c_hw[:m, :n]


In [40]:
# Load weights
meta = np.load(os.path.join(WEIGHT_DIR, 'meta.npy'), allow_pickle=True).item()
D = int(meta['embed_dim'])
H = int(meta['num_heads'])
HEAD_DIM = D // H
NUM_BLOCKS = int(meta['num_blocks'])
print('[INFO] D=', D, 'H=', H, 'HEAD_DIM=', HEAD_DIM, 'NUM_BLOCKS=', NUM_BLOCKS)

patch_w = np.load(os.path.join(WEIGHT_DIR, 'patch_embed_weight_fp32.npy'))
patch_b = np.load(os.path.join(WEIGHT_DIR, 'patch_embed_bias_fp32.npy'))
cls_token = np.load(os.path.join(WEIGHT_DIR, 'cls_token_fp32.npy'))
pos_embed = np.load(os.path.join(WEIGHT_DIR, 'pos_embed_fp32.npy'))

norm_w = np.load(os.path.join(WEIGHT_DIR, 'norm_weight_fp32.npy'))
norm_b = np.load(os.path.join(WEIGHT_DIR, 'norm_bias_fp32.npy'))
head_w = np.load(os.path.join(WEIGHT_DIR, 'head_weight_fp32.npy'))
head_b = np.load(os.path.join(WEIGHT_DIR, 'head_bias_fp32.npy'))


[INFO] D= 192 H= 3 HEAD_DIM= 64 NUM_BLOCKS= 12


In [41]:
def load_block_weights(i):
    idx = f'blk{i:02d}'
    w = {}
    w['ln1_w'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_ln1_weight_fp32.npy'))
    w['ln1_b'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_ln1_bias_fp32.npy'))
    w['ln2_w'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_ln2_weight_fp32.npy'))
    w['ln2_b'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_ln2_bias_fp32.npy'))

    w['qkv_w_i8'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_qkv_weight_int8.npy'))
    w['qkv_w_scale'] = float(np.load(os.path.join(WEIGHT_DIR, f'{idx}_qkv_weight_scale.npy')))
    w['qkv_b'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_qkv_bias_fp32.npy'))

    w['proj_w_i8'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_proj_weight_int8.npy'))
    w['proj_w_scale'] = float(np.load(os.path.join(WEIGHT_DIR, f'{idx}_proj_weight_scale.npy')))
    w['proj_b'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_proj_bias_fp32.npy'))

    w['mlp1_w_i8'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_mlp1_weight_int8.npy'))
    w['mlp1_w_scale'] = float(np.load(os.path.join(WEIGHT_DIR, f'{idx}_mlp1_weight_scale.npy')))
    w['mlp1_b'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_mlp1_bias_fp32.npy'))

    w['mlp2_w_i8'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_mlp2_weight_int8.npy'))
    w['mlp2_w_scale'] = float(np.load(os.path.join(WEIGHT_DIR, f'{idx}_mlp2_weight_scale.npy')))
    w['mlp2_b'] = np.load(os.path.join(WEIGHT_DIR, f'{idx}_mlp2_bias_fp32.npy'))
    return w


In [42]:
# Image preprocess (PIL optional)
if USE_IMAGE_NPY:
    img_chw = np.load(IMAGE_NPY).astype(np.float32)
    print('[OK] Loaded image tensor from npy:', img_chw.shape)
else:
    try:
        from PIL import Image
    except Exception as e:
        raise RuntimeError('PIL not available, set USE_IMAGE_NPY=True and provide image_fp32.npy')

    def resize_shorter(img, size):
        w, h = img.size
        if w < h:
            new_w = size
            new_h = int(h * size / w)
        else:
            new_h = size
            new_w = int(w * size / h)
        return img.resize((new_w, new_h))

    def center_crop(img, size):
        w, h = img.size
        left = (w - size) // 2
        top = (h - size) // 2
        return img.crop((left, top, left + size, top + size))

    img = Image.open(IMAGE_PATH).convert('RGB')
    img = resize_shorter(img, 256)
    img = center_crop(img, 224)
    arr = np.asarray(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (arr - mean) / std
    img_chw = np.transpose(arr, (2, 0, 1))
    print('[OK] Preprocessed image with PIL:', img_chw.shape)


[OK] Preprocessed image with PIL: (3, 224, 224)


In [43]:
print('[STEP] Patch embedding (numpy)')
# patch_w: [D, 3, 16, 16]
D = int(meta['embed_dim'])
patch_w2 = patch_w.reshape(D, -1)  # [D, 3*16*16]

# unfold patches
# unfold patches
img_C, img_H, img_W = img_chw.shape
ps = 16
nH = img_H // ps
nW = img_W // ps
patches = []
for i in range(nH):
    for j in range(nW):
        patch = img_chw[:, i*ps:(i+1)*ps, j*ps:(j+1)*ps].reshape(-1)
        patches.append(patch)
patches = np.stack(patches, axis=0)  # [num_patches, 3*16*16]
tokens = patches @ patch_w2.T + patch_b  # [num_patches, D]

# add cls token + pos
cls = cls_token.reshape(1, D)
tokens = np.concatenate([cls, tokens], axis=0)
tokens = tokens + pos_embed.reshape(tokens.shape)
print('[OK] tokens shape =', tokens.shape)


[STEP] Patch embedding (numpy)
[OK] tokens shape = (197, 192)


In [44]:
print('[STEP] Inference start')
x = tokens
for i in range(min(NUM_BLOCKS, MAX_BLOCKS)):
    t_block = time.time()
    print(f'[BLOCK {i:02d}] start')
    w = load_block_weights(i)

    # LN1
    t0 = time.time()
    x1 = layernorm_ps(x, w['ln1_w'], w['ln1_b'])
    print(f'  LN1 done, dt={time.time()-t0:.3f}s')

    # QKV (PL)
    t0 = time.time()
    a_q, scale_a = quantize_act(x1)
    qkv_i8 = run_gemm_pl_v2(a_q, w['qkv_w_i8'].T, scale_a, w['qkv_w_scale'], scale_a, timeout=TIMEOUT)
    qkv_fp = dequantize_int8(qkv_i8, scale_a) + w['qkv_b']
    print(f'  QKV GEMM done, dt={time.time()-t0:.3f}s')

    q = qkv_fp[:, :D]
    k = qkv_fp[:, D:2*D]
    v = qkv_fp[:, 2*D:]

    qh = q.reshape(-1, H, HEAD_DIM).transpose(1, 0, 2)
    kh = k.reshape(-1, H, HEAD_DIM).transpose(1, 0, 2)
    vh = v.reshape(-1, H, HEAD_DIM).transpose(1, 0, 2)

    attn_out = []
    for h in range(H):
        print(f'  [HEAD {h}] QK^T')
        qh_h = qh[h]
        kh_h = kh[h]
        vh_h = vh[h]

        q_i8, s_q = quantize_act(qh_h)
        k_i8, s_k = quantize_act(kh_h)
        score_i8 = run_gemm_pl_v2(q_i8, k_i8.T, s_q, s_k, s_q, timeout=TIMEOUT)
        score_fp = dequantize_int8(score_i8, s_q)
        score_fp = score_fp / np.sqrt(float(HEAD_DIM))
        score_sm = softmax_ps(score_fp)
        s_q_i8 = np.clip(np.rint(score_sm * 127.0), 0, 127).astype(np.int8)

        print(f'  [HEAD {h}] SV')
        v_i8, s_v = quantize_act(vh_h)
        sv_i8 = run_gemm_pl_v2(s_q_i8, v_i8, 1.0/127.0, s_v, s_v, timeout=TIMEOUT)
        sv_fp = dequantize_int8(sv_i8, s_v)
        attn_out.append(sv_fp)

    attn = np.concatenate(attn_out, axis=1)

    # Proj (PL)
    t0 = time.time()
    attn_i8, s_attn = quantize_act(attn)
    proj_i8 = run_gemm_pl_v2(attn_i8, w['proj_w_i8'].T, s_attn, w['proj_w_scale'], s_attn, timeout=TIMEOUT)
    proj_fp = dequantize_int8(proj_i8, s_attn) + w['proj_b']
    print(f'  Proj GEMM done, dt={time.time()-t0:.3f}s')

    x = x + proj_fp

    # LN2
    t0 = time.time()
    x2 = layernorm_ps(x, w['ln2_w'], w['ln2_b'])
    print(f'  LN2 done, dt={time.time()-t0:.3f}s')

    # MLP1 (PL)
    t0 = time.time()
    x2_i8, s_x2 = quantize_act(x2)
    mlp1_i8 = run_gemm_pl_v2(x2_i8, w['mlp1_w_i8'].T, s_x2, w['mlp1_w_scale'], s_x2, timeout=TIMEOUT)
    mlp1_fp = dequantize_int8(mlp1_i8, s_x2) + w['mlp1_b']
    mlp1_fp = gelu_ps(mlp1_fp)
    print(f'  MLP1 GEMM+GELU done, dt={time.time()-t0:.3f}s')

    # MLP2 (PL)
    t0 = time.time()
    mlp1_i8b, s_m1 = quantize_act(mlp1_fp)
    mlp2_i8 = run_gemm_pl_v2(mlp1_i8b, w['mlp2_w_i8'].T, s_m1, w['mlp2_w_scale'], s_m1, timeout=TIMEOUT)
    mlp2_fp = dequantize_int8(mlp2_i8, s_m1) + w['mlp2_b']
    print(f'  MLP2 GEMM done, dt={time.time()-t0:.3f}s')

    x = x + mlp2_fp

    print(f'[BLOCK {i:02d}] done, dt={time.time()-t_block:.3f}s')

print('[STEP] Final norm + head')
x = layernorm_ps(x, norm_w, norm_b)
logits = x[0] @ head_w.T + head_b
probs = np.exp(logits - np.max(logits))
probs = probs / np.sum(probs)
top5 = np.argsort(-probs)[:5]
print('[RESULT] Top-5 indices:', top5.tolist())
print('[RESULT] Top-5 probs:', probs[top5].tolist())


[STEP] Inference start
[BLOCK 00] start
  LN1 done, dt=0.007s
  QKV GEMM done, dt=14.996s
  [HEAD 0] QK^T
  [HEAD 0] SV
  [HEAD 1] QK^T
  [HEAD 1] SV
  [HEAD 2] QK^T
  [HEAD 2] SV
  Proj GEMM done, dt=4.969s
  LN2 done, dt=0.006s
  MLP1 GEMM+GELU done, dt=19.787s
  MLP2 GEMM done, dt=17.854s
[BLOCK 00] done, dt=71.330s
[BLOCK 01] start
  LN1 done, dt=0.007s
  QKV GEMM done, dt=14.516s
  [HEAD 0] QK^T
  [HEAD 0] SV
  [HEAD 1] QK^T
  [HEAD 1] SV
  [HEAD 2] QK^T
  [HEAD 2] SV
  Proj GEMM done, dt=5.243s
  LN2 done, dt=0.007s
  MLP1 GEMM+GELU done, dt=19.286s
  MLP2 GEMM done, dt=17.748s
[BLOCK 01] done, dt=70.306s
[BLOCK 02] start
  LN1 done, dt=0.007s
  QKV GEMM done, dt=14.724s
  [HEAD 0] QK^T
  [HEAD 0] SV
  [HEAD 1] QK^T
  [HEAD 1] SV
  [HEAD 2] QK^T
  [HEAD 2] SV
  Proj GEMM done, dt=5.001s
  LN2 done, dt=0.007s
  MLP1 GEMM+GELU done, dt=19.165s
  MLP2 GEMM done, dt=17.488s
[BLOCK 02] done, dt=69.788s
[BLOCK 03] start
  LN1 done, dt=0.006s
  QKV GEMM done, dt=14.649s
  [HEAD 0] QK^T
